**2D**
1) Define the domain $\Omega$
2) Generate triangulations T_h
3) Construct finite element space V_h
4) Get local stiffness matrix A_k
5) Get local mass matrix M_k
6) Assemble local stiffness matrices into global stiffness matrix A
7) Assemble local mass matrices into a global mass matrix M
8) Impose Dirichlet Boundary Condition
9) Solve $A * u = \lambda * M * u$
10) Extract eigenvalues
11) compute the eigenvectors
12) Compare with exact solution


In [67]:
# 2D let the domain be uniform and defined on [0, 1]x[0, 1]
import numpy as np
from scipy.linalg import eigh
from scipy.spatial import Delaunay
import pandas as pd
import matplotlib.pyplot as plt
plt.style.use("seaborn-v0_8-poster")

In [78]:
# method for putting values on the main diagonal
def put_value_in_special_index(matrix, grad_phi_i, grad_phi_j, index):
       # find the dot product of 2 gradients
       grad_phi_i_j = np.dot(grad_phi_i, grad_phi_j)
       
       matrix.put(index, grad_phi_i_j)

       return matrix

# stiffness matrix
def stiffness_matrix_A(grad_phi):
       # create a matrix (interior_nodes x interior_nodes) of zeros
       A_lower_tri = np.zeros((3, 3), dtype=float)
       # off diagonal values
       # first make lower triangular matrix
       # A_2_1
       put_value_in_special_index(A_lower_tri, grad_phi[1], grad_phi[0], 3)
       # A_3_1
       put_value_in_special_index(A_lower_tri, grad_phi[2], grad_phi[0], 6)
       # A_3_2
       put_value_in_special_index(A_lower_tri, grad_phi[2], grad_phi[1], 7)

       A = create_symmetric_matrix(A_lower_tri)
       # diagonal values
       put_value_in_special_index(A, grad_phi[0], grad_phi[0], 0)
       put_value_in_special_index(A, grad_phi[1], grad_phi[1], 4)
       put_value_in_special_index(A, grad_phi[2], grad_phi[2], 8)
       return A

def create_symmetric_matrix(lower_tri_matrix):
       # transpose the lower triangular matrix
       lower_tri_matrix_T = lower_tri_matrix.T
       # create symmetric matrix by adding the lower triangular matrix to its transpose
       sym_matrix = lower_tri_matrix + lower_tri_matrix_T

       return sym_matrix

# mass matrix
def M():
       
       return np.array(
              [
                     [2, 1, 1],
                     [1, 2, 1],
                     [1, 1, 2]
              ]
       )

# error of the first eigenvalue
# will get only the first elements of the arrays
def first_eigval_error(real_eigval, comp_eigval):
       error = abs(real_eigval - comp_eigval)
       return error

In [ ]:

# generate 2D domain with nodes P0, P1, P2, P3
# (if starting from the origin anticlockwise the diagonal will be between p1 and p3)
# (if starting from origin anticlockwise with nodes P0, P1, P4, P3, the diagonal will be between P0 and P3)
# same with P1 at (0, 1) if just clockwise, the diagonal is between P1 and P3
# if clockwise with nodes P0, P1, P3, P2 the diagonal is between P0 and P3
domain = np.array(
       [[0, 0], [1, 0],
       [1, 1], [0, 1]]
)
# create triangles on the domain
tri = Delaunay(domain)
print(tri.simplices)

# visualise the triangulations
plt.triplot(domain[:,0], domain[:,1], tri.simplices.copy())
plt.plot(domain[:,0], domain[:,1], "o")
plt.show()

# access the first triangle
triangle = tri.simplices[0]
# get coordinates of the triangle
coords = domain[triangle]
print(coords)

# find the area of the triangle
col = np.array([1, 1, 1])
# create 3x3 matrix to find the area
coords_matrix = np.hstack((coords, np.atleast_2d(col).T))

# area of a triangle
T_k = 0.5 * np.linalg.det(coords_matrix)
print(T_k)

x = []
y = []
# for coordinate in all of the coordinates of the nodes of this triangle
for coord in coords:
       x.append(float(coord[0]))
       y.append(float(coord[1]))

c = []   
c.append(x[2] - x[1])
c.append(x[0] - x[2])
c.append(x[1] - x[0])

b = []
b.append(y[1] - y[2])
b.append(y[2] - y[0])
b.append(y[0] - y[1])

# find the gradients of the basis functions
grad_phi = np.array([
       (1 / (2 * T_k)) * np.array([b_i, c_i]) for b_i, c_i in zip(b, c)
])

print(grad_phi)

# get local stiffness matrix
A_local = T_k * stiffness_matrix_A(grad_phi)
print(A_local)
M_local = (T_k / 12) * M()
print(M_local)